# Treinamento da dengue no Colab
Abra a aba **Arquivos** do Colab e envie para `/content` o `manifesto.json`, `codigo_e_atributos.zip` e **ou** o Parquet completo `dengue-analitico.parquet` **ou** todas as partes `.part001`–`.part005`. Execute as células em ordem. O treino mostra cada modelo, métricas e tempo restante; a última célula baixa os resultados. `/content` é temporário.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, sys, zipfile
PACOTE = Path('/content')
if not (PACOTE / 'manifesto.json').is_file() and (PACOTE / 'pacoteColab/manifesto.json').is_file():
    PACOTE = PACOTE / 'pacoteColab'
PROJETO = Path('/content/denguinhos')
if not (PACOTE / 'manifesto.json').is_file():
    raise FileNotFoundError('Envie manifesto.json e codigo_e_atributos.zip pela aba Arquivos do Colab.')
MANIFESTO = json.loads((PACOTE / 'manifesto.json').read_text(encoding='utf-8'))
print('Partes:', len(MANIFESTO['dataset']['partes']), '| Python:', sys.version.split()[0])
if not (3, 12) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError('Este pacote foi preparado para Python 3.12 ou 3.13 do Colab. Recrie o pacote se o runtime mudar de versão.')


In [ ]:
def digest(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b''):
            h.update(bloco)
    return h.hexdigest()
zip_local = PACOTE / MANIFESTO['codigo']['arquivo']
assert digest(zip_local) == MANIFESTO['codigo']['sha256'], 'ZIP do código corrompido.'
PROJETO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_local) as pacote:
    pacote.extractall(PROJETO)
destino = PROJETO / MANIFESTO['dataset']['destino']
destino.parent.mkdir(parents=True, exist_ok=True)
parquet_enviado = PACOTE / 'dengue-analitico.parquet'
if parquet_enviado.is_file():
    assert parquet_enviado.stat().st_size == MANIFESTO['dataset']['bytes'], 'Tamanho do Parquet divergente.'
    assert digest(parquet_enviado) == MANIFESTO['dataset']['sha256'], 'Parquet enviado corrompido.'
    if destino != parquet_enviado and not destino.exists():
        destino.symlink_to(parquet_enviado)  # evita duplicar 595 MiB no disco
else:
    temporario = destino.with_suffix('.parquet.part')
    h_total = hashlib.sha256()
    with open(temporario, 'wb') as saida:
        for parte in MANIFESTO['dataset']['partes']:
            h_parte = hashlib.sha256()
            caminho = PACOTE / parte['arquivo']
            with open(caminho, 'rb') as entrada:
                while True:
                    bloco = entrada.read(1024 * 1024)
                    if not bloco: break
                    saida.write(bloco)
                    h_parte.update(bloco)
                    h_total.update(bloco)
            assert h_parte.hexdigest() == parte['sha256'], f"Parte corrompida: {parte['arquivo']}"
            print('Verificada:', parte['arquivo'])
    assert h_total.hexdigest() == MANIFESTO['dataset']['sha256'], 'Parquet final divergente.'
    os.replace(temporario, destino)
# O contrato de dados contém o caminho absoluto do Mac: atualize só o caminho no Colab.
schema_path = PROJETO / 'treinamentoDoModelo/artefatos/ouro/schema_atributos.json'
schema = json.loads(schema_path.read_text(encoding='utf-8'))
schema['config_preparo']['entrada'] = str(destino)
schema_path.write_text(json.dumps(schema, ensure_ascii=False, indent=2), encoding='utf-8')
print('Dataset completo reconstruído:', destino, f'{destino.stat().st_size / 2**20:.1f} MiB')


In [ ]:
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(PROJETO / 'treinamentoDoModelo/requirements-colab.txt')], check=True)
import yaml
config_path = PROJETO / 'treinamentoDoModelo/config.yaml'
config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
config['treino']['progresso']['mostrar'] = True
config['treino']['progresso']['intervalo_arvores'] = 50
config_path.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding='utf-8')
import pyarrow.parquet as pq
atributos = PROJETO / 'treinamentoDoModelo/artefatos/ouro/atributos.parquet'
n_linhas = pq.ParquetFile(atributos).metadata.num_rows
t = config['treino']
print(f'Painel: {n_linhas:,} linhas | validação: {t["validacao_anos"]} | horizontes: {t["horizontes"]}')
print(f'Plano: {len(t["candidatos"])} modelos × {len(t["horizontes"])} horizontes × {len(t["validacao_anos"])} ano(s) = {len(t["candidatos"]) * len(t["horizontes"]) * len(t["validacao_anos"])} ajustes')
print('Modelos:', ', '.join(t['candidatos']))


In [ ]:
# Saída em tempo real. O pulso de 30 s mantém visível o modelo em execução.
import threading, time
inicio = time.monotonic()
ultimo_status = {'texto': 'iniciando processo'}
parar_pulso = threading.Event()
def pulso():
    while not parar_pulso.wait(30):
        print(f'[Colab] {int(time.monotonic() - inicio)} s decorridos | {ultimo_status["texto"]}', flush=True)
threading.Thread(target=pulso, daemon=True).start()
processo = subprocess.Popen([sys.executable, '-u', str(PROJETO / 'treinamentoDoModelo/executar.py'), 'treinar'], cwd=PROJETO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
try:
    for linha in processo.stdout:
        print(linha, end='', flush=True)
        if ' INÍCIO ' in linha or ' FIM ' in linha or 'Horizonte h' in linha:
            ultimo_status['texto'] = linha.strip()[-160:]
    codigo_saida = processo.wait()
except KeyboardInterrupt:
    processo.terminate()
    codigo_saida = processo.wait()
    print('Treino interrompido; os artefatos já salvos ainda podem ser baixados.')
finally:
    parar_pulso.set()
print('Código de saída:', codigo_saida, '| duração:', round((time.monotonic() - inicio) / 60, 1), 'minutos')
if codigo_saida != 0: print('Confira o traceback acima e baixe os artefatos parciais na célula seguinte.')


In [ ]:
ARTEFATOS = PROJETO / 'treinamentoDoModelo/artefatos'
pastas = sorted((ARTEFATOS / 'modelos').glob('20*'))
if not pastas: raise FileNotFoundError('Nenhum artefato de treino foi criado.')
pasta = pastas[-1]
print('Experimento mais recente:', pasta, '| concluído:', (pasta / 'resumo.json').exists())
import pandas as pd
linhas = []
for relatorio in sorted(pasta.glob('validacao_h*.json')):
    for item in json.loads(relatorio.read_text()):
        if item.get('grupo') != 'conhecidos': continue
        m = item.get('metricas', {})
        o = m.get('ocorrencia', {})
        linhas.append({'horizonte': relatorio.stem.split('_')[-1], 'modelo': item['modelo'],
                       'recall': o.get('recall'), 'precisão': o.get('precision'),
                       'PR-AUC': o.get('pr_auc'), 'MAE macro': m.get('macro_mae'),
                       'segundos': item.get('segundos_treino'), 'erro': item.get('erro')})
if linhas:
    print(pd.DataFrame(linhas).sort_values(['horizonte', 'recall'], ascending=[True, False]).to_string(index=False))
if (pasta / 'resumo.json').exists():
    print('Vencedores:', json.dumps({h: r['candidato'] for h, r in json.loads((pasta / 'resumo.json').read_text()).items()}, ensure_ascii=False))
# Não incluir o Parquet original no download: ele já está no Mac.
arquivo_saida = Path('/content/resultados_treinamento.zip')
with zipfile.ZipFile(arquivo_saida, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as z:
    for camada in ('modelos', 'relatorios'):
        for arq in (ARTEFATOS / camada).rglob('*'):
            if arq.is_file(): z.write(arq, arcname=str(arq.relative_to(ARTEFATOS)))
print('Resultados prontos para download:', arquivo_saida, f'{arquivo_saida.stat().st_size / 2**20:.1f} MiB')
from google.colab import files
files.download(str(arquivo_saida))
